In [1]:
import google.generativeai as genai
import pandas as pd
import time
import logging
from tqdm import tqdm

# Set up Gemini API
genai.configure(api_key="API KEY")
model = genai.GenerativeModel("moodel you want to use")

In [2]:
# Assign experts for each target
target_role_map = {
    "Atheism": "theologian",
    "Climate Change is a Real Concern": "environmental scientist",
    "Feminist Movement": "sociologist",
    "Hillary Clinton": "political scientist",
    "Legalization of Abortion": "sociologist",
    "Donald Trump": "political scientist"
}

In [3]:
def load_csv_data(file_path):
    encodings = ['utf-8', 'latin1', 'ISO-8859-1']
    for enc in encodings:
        try:
            return pd.read_csv(file_path, encoding=enc, engine='python')
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Unable to read {file_path} with encodings: {', '.join(encodings)}")


In [4]:
def get_completion_with_role(role, instruction, content):
    for _ in range(5):  # Retry up to 5 times
        try:
            response = model.generate_content(f"You are a {role}.\n{instruction}\n{content}")
            return response.text
        except Exception as e:
            logging.error(f"Error: {str(e)}")
            time.sleep(2)
    return "Error"


In [5]:
def linguist_analysis(tweet):
    instruction = "Explain linguistic elements in the sentence, including grammar, rhetorical devices, and lexical choices."
    return get_completion_with_role("linguist", instruction, tweet)

def expert_analysis(tweet, target):
    role = target_role_map.get(target, "expert")
    instruction = f"Explain the key elements in the quote, including characters, events, or entities, and their relation to {target}."
    return get_completion_with_role(role, instruction, tweet)

def user_analysis(tweet):
    instruction = "Analyze content, hashtags, slang, emotional tone, and implied meaning."
    return get_completion_with_role("heavy social media user", instruction, tweet)

def stance_analysis(tweet, ling_response, expert_response, user_response, target, stance):
    role = target_role_map.get(target, "expert")
    prompt = f"'''{tweet}'''\n <<< {ling_response} >>>\n [[[ {expert_response} ]]]\n--- {user_response} ---\n\
              You think the attitude behind the sentence is {stance} towards {target}. Identify the best supporting evidence and explain why."
    return get_completion_with_role(role, prompt, "")

def final_judgement(tweet, favor_response, against_response, target):
    prompt = f"Determine if the sentence supports or opposes {target}, or if it's irrelevant.\nSentence: {tweet}\n\
               Favor Arguments: {favor_response}\nAgainst Arguments: {against_response}\n\
               Choose: A (Against), B (Favor), C (Irrelevant). Return only the option."
    return get_completion_with_role("judge", prompt, "")


In [6]:
def add_predictions_sequential(data):
    results = []
    
    with tqdm(total=len(data), desc="Processing Tweets", unit="tweet") as pbar:
        for _, row in data.iterrows():
            tweet, target = row['Tweet'], row['Target']
            ling_response = linguist_analysis(tweet)
            expert_response = expert_analysis(tweet, target)
            user_response = user_analysis(tweet)
            favor_response = stance_analysis(tweet, ling_response, expert_response, user_response, target, "in favor")
            against_response = stance_analysis(tweet, ling_response, expert_response, user_response, target, "against")
            final_response = final_judgement(tweet, favor_response, against_response, target)

            results.append({
                'Tweet': tweet, 'Target': target, 'Linguist Analysis': ling_response,
                'Expert Analysis': expert_response, 'User Analysis': user_response,
                'In Favor': favor_response, 'Against': against_response, 'Final Judgement': final_response
            })
            
            pbar.update(1)  # Update progress bar after each row

    for idx, res in enumerate(results):
        for key, value in res.items():
            data.at[idx, key] = value

In [ ]:
data = load_csv_data("File path")
add_predictions_sequential(dataa)
dataa.to_csv("Data path", index=False)